In [22]:
import random
import statistics
import matplotlib.pyplot as plt

"""
We apply Thompson Sampling to a famous example of the MABP - slot machines.

We create an object for the slot machines, in which each slot machine is assigned a random success rate.
"""

class SlotMachines:
    
    def __init__(self, conversions):
        
        self.conversions = conversions
        
        self.n = len(conversions)
    
    def pull_arm(self, slot_machine):
        
        return random.random() <= self.conversions[slot_machine]
    

"""
We create a Thompson class for our simulation.
"""

class Thompson:
    
    def __init__(self, slot_machines):
        
        self.slot_machines = slot_machines
        
        self.rewards = [0 for i in range(slot_machines.n)]
        self.penalties = [0 for i in range(slot_machines.n)]
        
        self.picks = [] # Store the picks for each round.
        
        self.total_reward = 0 # Keep track of the total reward
    
    def round(self):
        
        winner, highest_estimate = None, float('-inf')
        
        for i in range(self.slot_machines.n):
            estimate = random.betavariate(self.rewards[i] + 1, self.penalties[i] + 1) # Randomly sample the beta distribution of a choice.
            
            if estimate > highest_estimate: # If the sample is higher than the current highest, update the highest estimate.
                
                highest_estimate = estimate
                winner = i
            
        reward = self.slot_machines.pull_arm(winner) # Pull the arm of the winning slot machine.
        
        if reward: # If it's a success, you get a reward.
            self.rewards[winner] += 1
        else:
            self.penalties[winner] += 1
        
        self.picks.append(winner)
        
        self.total_reward += reward
    
    def run_thompson(self, rounds):
        
        for i in range(rounds):
            self.round()
        
"""
Create a class for random selection to compare the results of Thompson sampling.

In random selection, we just randomly select an ad in each round.
"""

class RandomSelection:
    
    def __init__(self, slot_machines):
        
        self.slot_machines = slot_machines
        
        self.total_reward = 0
        
        self.picks = []
    
    def round(self):
        
        random_pick = random.randint(0, self.slot_machines.n - 1) # Randomly select an ad.
        
        self.total_reward += self.slot_machines.pull_arm(random_pick) # Pull the arm of the ad and update the total reward.
        
        self.picks.append(random_pick)
        
    def run_random(self, trials):
        
        for i in range(trials):
            self.round()


def trial(): # Simulate a run of Thompson sampling and random selection
    
    n = 5
    trials = 20000
    
    conversions = [(random.random() / 5) for i in range(n)] # Generate conversion rates
    
    best = conversions.index(max(conversions)) # Find the index of the slot machine with the highest conversion rate
    
    slot_machines = SlotMachines(conversions) # Initialize the slot machines.
    
    thompson_selection = Thompson(slot_machines)
    random_selection = RandomSelection(slot_machines)
    
    # Run the simulations
    
    thompson_selection.run_thompson(trials)
    random_selection.run_random(trials)
    
    return [thompson_selection.total_reward, random_selection.total_reward, (thompson_selection.total_reward / (random_selection.total_reward + 1)), best == statistics.mode(thompson_selection.picks)]
def main(): # Execute multiple runs to accurately compare the performance of TS to random selection.
    
    ratios = []
    
    thompson_rewards = []
    random_rewards = []
    
    successes = 0
    total = 0
    
    trials = 100
    
    for i in range(trials):
        
        result = trial()
        
        thompson_rewards.append(result[0])
        random_rewards.append(result[1])
        
        ratios.append(result[2])
        
        successes += result[3]
        total += 1
    
    # Summarize the results
    print(f"On average, the reward attained by Thompson sampling was {round(statistics.mean(ratios), 2)} times as large as random selection.")
    print(f"On average, Thompson sampling scored a reward of: {round(statistics.mean(thompson_rewards), 2)}")
    print(f"On average, random selection scored a reward of: {round(statistics.mean(random_rewards), 2)}")
    print(f"Thompson Sampling achieved a success rate of {round(successes / total, 2)} at predicting the slot machine with the highest success rate.")

if __name__ == '__main__':
    main()

On average, the reward attained by Thompson sampling was 1.65 times as large as random selection.
On average, Thompson sampling scored a reward of: 3290.96
On average, random selection scored a reward of: 2096.35
Thompson Sampling achieved a success rate of 0.91 at predicting the slot machine with the highest success rate.
